## Example 1: send2trash functionality test 1

In [ ]:
# Example 1: Basic Send2Trash Functionality and Cross-Platform Support
import send2trash
import tempfile
import os
import platform

print("=== Basic Send2Trash Functionality ===")

# Check platform compatibility
print(f"Operating system: {platform.system()}")
print(f"Platform: {platform.platform()}")

# Create a temporary test file
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as temp_file:
    temp_file.write("This is a test file for send2trash demonstration.")
    temp_path = temp_file.name

print(f"Created temporary test file: {os.path.basename(temp_path)}")
print(f"File exists: {os.path.exists(temp_path)}")
print(f"File size: {os.path.getsize(temp_path)} bytes")

# Demonstrate send2trash functionality
print(f"\n=== Send2Trash Operation ===")

try:
    # Send file to trash/recycle bin
    send2trash.send2trash(temp_path)
    print(f"✓ File sent to trash successfully")
    
    # Verify file is no longer in original location
    print(f"File exists after trash: {os.path.exists(temp_path)}")
    
except Exception as e:
    print(f"✗ Error sending file to trash: {e}")
    # Cleanup in case of error
    if os.path.exists(temp_path):
        os.unlink(temp_path)

# Show send2trash module information
print(f"\n=== Module Information ===")
print(f"Send2trash module: {send2trash.__name__}")
print(f"Available functions: {[attr for attr in dir(send2trash) if not attr.startswith('_')]}")

# Cross-platform behavior
print(f"\n=== Cross-Platform Behavior ===")
platform_behaviors = {
    'Windows': 'Moves files to Recycle Bin',
    'Darwin': 'Moves files to Trash (macOS)',
    'Linux': 'Moves files to user trash directory'
}

current_platform = platform.system()
if current_platform in platform_behaviors:
    print(f"On {current_platform}: {platform_behaviors[current_platform]}")
else:
    print(f"Platform {current_platform}: Uses appropriate system trash mechanism")

# Safety considerations
print(f"\n=== Safety Considerations ===")
print("✓ Files can typically be recovered from trash/recycle bin")
print("✓ Safer than permanent deletion with os.remove()")
print("✓ Respects system trash policies and permissions")
print("⚠ Files may be permanently deleted if trash is emptied")

print("Basic send2trash functionality completed!")

## Example 2: send2trash functionality test 2

In [ ]:
# Example 2: Multiple Files and Directory Handling
import send2trash
import tempfile
import os
import shutil

print("=== Multiple Files and Directory Handling ===")

# Create multiple test files
test_files = []
for i in range(3):
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix=f'_test_{i}.txt') as temp_file:
        temp_file.write(f"Test file content {i}")
        test_files.append(temp_file.name)

print(f"Created {len(test_files)} test files:")
for i, file_path in enumerate(test_files):
    print(f"  {i+1}. {os.path.basename(file_path)} ({os.path.getsize(file_path)} bytes)")

# Create a test directory with files
test_dir = tempfile.mkdtemp(prefix='send2trash_test_')
subdir = os.path.join(test_dir, 'subdirectory')
os.makedirs(subdir)

# Create files in directory
dir_files = []
for i in range(2):
    file_path = os.path.join(test_dir, f'dir_file_{i}.txt')
    with open(file_path, 'w') as f:
        f.write(f"Directory file {i}")
    dir_files.append(file_path)

# Create file in subdirectory
subdir_file = os.path.join(subdir, 'nested_file.txt')
with open(subdir_file, 'w') as f:
    f.write("Nested directory file")

print(f"\nCreated test directory: {os.path.basename(test_dir)}")
print(f"Directory contains {len(dir_files)} files and 1 subdirectory")

# Send multiple files to trash
print(f"\n=== Sending Multiple Files to Trash ===")

try:
    # Send individual files
    for i, file_path in enumerate(test_files):
        send2trash.send2trash(file_path)
        print(f"✓ File {i+1} sent to trash: {os.path.basename(file_path)}")
    
    # Verify files are gone
    remaining_files = [f for f in test_files if os.path.exists(f)]
    print(f"Files remaining after trash operation: {len(remaining_files)}")
    
except Exception as e:
    print(f"✗ Error sending files to trash: {e}")
    # Cleanup remaining files
    for file_path in test_files:
        if os.path.exists(file_path):
            os.unlink(file_path)

# Send directory to trash
print(f"\n=== Sending Directory to Trash ===")

try:
    # Count items before deletion
    total_items = len(os.listdir(test_dir)) + len(os.listdir(subdir))
    print(f"Directory contains {total_items} total items")
    
    # Send entire directory to trash
    send2trash.send2trash(test_dir)
    print(f"✓ Directory sent to trash: {os.path.basename(test_dir)}")
    
    # Verify directory is gone
    print(f"Directory exists after trash: {os.path.exists(test_dir)}")
    
except Exception as e:
    print(f"✗ Error sending directory to trash: {e}")
    # Cleanup in case of error
    if os.path.exists(test_dir):
        shutil.rmtree(test_dir)

# Batch operations simulation
print(f"\n=== Batch Operations Pattern ===")

def safe_send_to_trash(paths, continue_on_error=True):
    """Safely send multiple paths to trash with error handling"""
    results = {'success': [], 'failed': []}
    
    for path in paths:
        try:
            if os.path.exists(path):
                send2trash.send2trash(path)
                results['success'].append(path)
            else:
                results['failed'].append((path, 'File not found'))
        except Exception as e:
            results['failed'].append((path, str(e)))
            if not continue_on_error:
                break
    
    return results

# Demonstrate batch operation (with non-existent files)
sample_paths = [
    '/tmp/nonexistent1.txt',
    '/tmp/nonexistent2.txt', 
    '/tmp/nonexistent3.txt'
]

print(f"Testing batch operation with {len(sample_paths)} paths:")
batch_results = safe_send_to_trash(sample_paths)
print(f"  Successful: {len(batch_results['success'])}")
print(f"  Failed: {len(batch_results['failed'])}")

if batch_results['failed']:
    print("Failed operations:")
    for path, error in batch_results['failed'][:2]:  # Show first 2
        print(f"    {os.path.basename(path)}: {error}")

print("Multiple files and directory handling completed!")

## Example 3: send2trash functionality test 3

In [ ]:
# Example 3: Error Handling and Edge Cases
import send2trash
import tempfile
import os
import platform
import errno

print("=== Error Handling and Edge Cases ===")

# Test various error conditions
print("=== Error Condition Testing ===")

# 1. Non-existent file
print("\n1. Non-existent file handling:")
nonexistent_file = '/tmp/this_file_does_not_exist.txt'
try:
    send2trash.send2trash(nonexistent_file)
except FileNotFoundError as e:
    # send2trash refuses a path that does not exist: FileNotFoundError, errno ENOENT
    assert e.errno == errno.ENOENT, e
    print(f"✓ Non-existent file rejected with {type(e).__name__} (errno {e.errno})")
else:
    raise AssertionError(f"send2trash({nonexistent_file!r}) should raise FileNotFoundError")

# 2. Empty filename
print("\n2. Empty filename handling:")
try:
    send2trash.send2trash('')
except FileNotFoundError as e:
    # send2trash treats '' as a path that does not exist: FileNotFoundError, errno ENOENT
    assert e.errno == errno.ENOENT, e
    print(f"✓ Empty filename rejected with {type(e).__name__} (errno {e.errno})")
else:
    raise AssertionError("send2trash('') should raise FileNotFoundError")

# 3. Permission testing with temporary file
print("\n3. Permission and access testing:")

# Create a test file for permission testing
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as temp_file:
    temp_file.write("Permission test file")
    perm_test_file = temp_file.name

print(f"Created permission test file: {os.path.basename(perm_test_file)}")

# Test with different file states
file_states = [
    ('readable', os.path.exists(perm_test_file)),
    ('writable', os.access(perm_test_file, os.W_OK) if os.path.exists(perm_test_file) else False),
    ('executable', os.access(perm_test_file, os.X_OK) if os.path.exists(perm_test_file) else False)
]

for state, accessible in file_states:
    print(f"  File is {state}: {accessible}")

try:
    send2trash.send2trash(perm_test_file)
except OSError:
    if os.path.exists(perm_test_file):
        os.unlink(perm_test_file)  # Cleanup, then fail: a writable temp file must be trashable
    raise
assert not os.path.exists(perm_test_file), f"send2trash returned but {perm_test_file} is still there"
print(f"✓ Permission test file sent to trash successfully")

# 4. Path validation and normalization
print("\n=== Path Validation and Normalization ===")

test_paths = [
    './relative_path.txt',
    '../parent_dir/file.txt',
    '~/home_file.txt',  
    '/absolute/path/file.txt'
]

print("Path normalization examples:")
for path in test_paths:
    normalized = os.path.normpath(os.path.expanduser(path))
    print(f"  Original: {path}")
    print(f"  Normalized: {normalized}")
    print(f"  Is absolute: {os.path.isabs(normalized)}")
    print()

# 5. Safe wrapper function with comprehensive error handling
print("=== Safe Wrapper Function ===")

def safe_send2trash(path, verbose=True):
    """Safely send file to trash: False for a missing or unwritable path, ValueError for a non-path input"""
    # Validate input first: a non-path argument is the caller's bug, so it raises instead of folding into False
    if not path or not isinstance(path, str):
        raise ValueError("Path must be a non-empty string")
    
    try:
        # Normalize path
        normalized_path = os.path.normpath(os.path.expanduser(path))
        
        # Check if path exists
        if not os.path.exists(normalized_path):
            if verbose:
                print(f"⚠ Warning: Path does not exist: {path}")
            return False
        
        # Check permissions
        parent_dir = os.path.dirname(normalized_path)
        if not os.access(parent_dir, os.W_OK):
            if verbose:
                print(f"⚠ Warning: No write permission for parent directory")
            return False
        
        # Send to trash
        send2trash.send2trash(normalized_path)
        if verbose:
            print(f"✓ Successfully sent to trash: {os.path.basename(path)}")
        return True
        
    except PermissionError as e:
        if verbose:
            print(f"✗ Permission denied: {e}")
        return False
    except OSError as e:
        if verbose:
            print(f"✗ OS error: {e}")
        return False

# Test safe wrapper with various inputs: a missing path returns False, a non-path input raises ValueError
test_inputs = [
    ('/nonexistent/file.txt', False),
    ('', ValueError),
    (None, ValueError),
    (123, ValueError),  # Non-string input
]

print("Testing safe wrapper function:")
for i, (test_input, expected) in enumerate(test_inputs, 1):
    print(f"\nTest {i}: {repr(test_input)}")
    if expected is ValueError:
        try:
            safe_send2trash(test_input)
        except ValueError as e:
            assert "non-empty string" in str(e), e
            print(f"  Rejected: {type(e).__name__}: {e}")
        else:
            raise AssertionError(f"safe_send2trash({test_input!r}) accepted a non-path input")
    else:
        result = safe_send2trash(test_input)
        print(f"  Result: {result}")
        assert result is expected, f"safe_send2trash({test_input!r}) returned {result!r}, expected {expected!r}"

# And a real file: the wrapper trashes it and returns True
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as temp_file:
    temp_file.write("Safe wrapper test file")
    wrapper_test_file = temp_file.name
print(f"\nTest {len(test_inputs) + 1}: a real file")
assert safe_send2trash(wrapper_test_file) is True, f"safe_send2trash refused a writable temp file {wrapper_test_file}"
assert not os.path.exists(wrapper_test_file), f"safe_send2trash returned True but {wrapper_test_file} is still there"

print("Error handling and edge cases completed!")

## Example 4: send2trash functionality test 4

In [ ]:
# Example 4: Performance and Efficiency Considerations
import send2trash
import tempfile
import os
import time
from contextlib import contextmanager

print("=== Performance and Efficiency Considerations ===")

# Performance measurement context manager
@contextmanager
def measure_time(operation_name):
    """Context manager to measure operation time"""
    start_time = time.time()
    try:
        yield
    finally:
        end_time = time.time()
        duration = end_time - start_time
        print(f"  {operation_name}: {duration:.4f} seconds")

# Create test files for performance testing
print("=== Performance Testing Setup ===")

test_files = []
file_sizes = [1024, 10240, 102400]  # 1KB, 10KB, 100KB

for i, size in enumerate(file_sizes):
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix=f'_perf_{i}.txt') as temp_file:
        # Write data of specified size
        content = 'A' * size
        temp_file.write(content)
        test_files.append((temp_file.name, size))

print(f"Created {len(test_files)} test files:")
for path, size in test_files:
    print(f"  {os.path.basename(path)}: {size:,} bytes")

# Test single file performance
print(f"\n=== Single File Performance ===")

for path, size in test_files:
    filename = os.path.basename(path)
    
    with measure_time(f"Trashing {filename} ({size:,} bytes)"):
        try:
            send2trash.send2trash(path)
            print(f"    ✓ Success")
        except Exception as e:
            print(f"    ✗ Error: {e}")
            # Cleanup if failed
            if os.path.exists(path):
                os.unlink(path)

# Batch operation performance
print(f"\n=== Batch Operation Performance ===")

# Create multiple small files for batch testing
batch_files = []
for i in range(10):
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix=f'_batch_{i}.txt') as temp_file:
        temp_file.write(f"Batch test file {i}")
        batch_files.append(temp_file.name)

print(f"Created {len(batch_files)} files for batch testing")

# Individual vs batch operations
print("\nComparing individual operations:")

with measure_time("Individual send2trash calls"):
    for path in batch_files:
        try:
            send2trash.send2trash(path)
        except Exception as e:
            print(f"    Error with {os.path.basename(path)}: {e}")
            if os.path.exists(path):
                os.unlink(path)

# Memory usage considerations
print(f"\n=== Memory Usage Considerations ===")

# Test with different file types and characteristics
test_scenarios = [
    {
        'name': 'Text files',
        'description': 'Small text files with minimal metadata',
        'overhead': 'Low memory overhead, fast operation'
    },
    {
        'name': 'Binary files', 
        'description': 'Binary files may have different handling',
        'overhead': 'Similar overhead, platform-dependent'
    },
    {
        'name': 'Large directories',
        'description': 'Directories with many files',
        'overhead': 'Higher overhead due to recursive operations'
    },
    {
        'name': 'Special files',
        'description': 'Symbolic links, device files, etc.',
        'overhead': 'Variable overhead, platform-specific'
    }
]

print("Memory usage characteristics by file type:")
for scenario in test_scenarios:
    print(f"  {scenario['name']}:")
    print(f"    Description: {scenario['description']}")
    print(f"    Overhead: {scenario['overhead']}")

# Best practices for performance
print(f"\n=== Performance Best Practices ===")

best_practices = [
    {
        'practice': 'Batch operations when possible',
        'benefit': 'Reduces system call overhead',
        'implementation': 'Group related files for deletion'
    },
    {
        'practice': 'Handle errors gracefully',
        'benefit': 'Prevents blocking on failed operations',
        'implementation': 'Use try-except blocks with continue_on_error'
    },
    {
        'practice': 'Validate paths before operation',
        'benefit': 'Avoids unnecessary system calls',
        'implementation': 'Check os.path.exists() first'
    },
    {
        'practice': 'Use absolute paths',
        'benefit': 'Avoids path resolution overhead',
        'implementation': 'Convert with os.path.abspath()'
    }
]

for practice in best_practices:
    print(f"  {practice['practice']}:")
    print(f"    Benefit: {practice['benefit']}")
    print(f"    Implementation: {practice['implementation']}")
    print()

# Efficiency comparison with os.remove()
print(f"=== Send2trash vs os.remove() Comparison ===")

comparison_aspects = {
    'Safety': {
        'send2trash': 'Files can be recovered from trash',
        'os.remove': 'Files permanently deleted immediately'
    },
    'Performance': {
        'send2trash': 'Slightly slower due to trash system interaction',
        'os.remove': 'Faster, direct file deletion'
    },
    'System Integration': {
        'send2trash': 'Integrates with system trash/recycle bin',
        'os.remove': 'Bypasses system trash entirely'
    },
    'Error Handling': {
        'send2trash': 'Platform-specific error handling',
        'os.remove': 'Standard OS error codes'
    }
}

for aspect, methods in comparison_aspects.items():
    print(f"{aspect}:")
    for method, description in methods.items():
        print(f"  {method}: {description}")
    print()

# Resource cleanup recommendations
print(f"=== Resource Cleanup Recommendations ===")
print("✓ Always handle exceptions when using send2trash")
print("✓ Provide fallback cleanup with os.unlink() for critical operations")
print("✓ Consider user preferences for trash vs permanent deletion")
print("✓ Monitor trash/recycle bin space in long-running applications")
print("✓ Test behavior across different platforms and file systems")

print("Performance and efficiency considerations completed!")

## Example 5: send2trash functionality test 5

In [ ]:
# Example 5: Production Integration and Best Practices
import send2trash
import tempfile
import os
import shutil
import logging
from pathlib import Path
from datetime import datetime

print("=== Production Integration and Best Practices ===")

# Production-ready file cleanup manager
class FileCleanupManager:
    """Production-ready file cleanup with send2trash integration"""
    
    def __init__(self, use_trash=True, log_operations=True):
        self.use_trash = use_trash
        self.log_operations = log_operations
        self.operations_log = []
        
        if log_operations:
            logging.basicConfig(level=logging.INFO)
            self.logger = logging.getLogger(__name__)
    
    def cleanup_file(self, path, force_permanent=False):
        """Cleanup single file with optional permanent deletion"""
        path = Path(path)
        operation = {
            'timestamp': datetime.now().isoformat(),
            'path': str(path),
            'method': None,
            'success': False,
            'error': None
        }
        
        try:
            if not path.exists():
                operation['error'] = 'File not found'
                return False
            
            if self.use_trash and not force_permanent:
                send2trash.send2trash(str(path))
                operation['method'] = 'send2trash'
                if self.log_operations:
                    self.logger.info(f"Sent to trash: {path}")
            else:
                if path.is_dir():
                    shutil.rmtree(str(path))
                else:
                    path.unlink()
                operation['method'] = 'permanent_delete'
                if self.log_operations:
                    self.logger.info(f"Permanently deleted: {path}")
            
            operation['success'] = True
            return True
            
        except Exception as e:
            operation['error'] = str(e)
            if self.log_operations:
                self.logger.error(f"Failed to cleanup {path}: {e}")
            return False
        finally:
            self.operations_log.append(operation)
    
    def cleanup_directory(self, directory, pattern='*', recursive=True):
        """Cleanup files in directory matching pattern"""
        directory = Path(directory)
        
        if not directory.exists():
            if self.log_operations:
                self.logger.warning(f"Directory not found: {directory}")
            return {'success': 0, 'failed': 0}
        
        results = {'success': 0, 'failed': 0}
        
        if recursive:
            files = directory.rglob(pattern)
        else:
            files = directory.glob(pattern)
        
        for file_path in files:
            if file_path.is_file():
                if self.cleanup_file(file_path):
                    results['success'] += 1
                else:
                    results['failed'] += 1
        
        return results
    
    def cleanup_temporary_files(self, temp_dir=None, max_age_hours=24):
        """Cleanup old temporary files"""
        temp_dir = Path(temp_dir or tempfile.gettempdir())
        current_time = datetime.now().timestamp()
        cutoff_time = current_time - (max_age_hours * 3600)
        
        results = {'success': 0, 'failed': 0, 'skipped': 0}
        
        for temp_file in temp_dir.glob('*'):
            if temp_file.is_file():
                try:
                    file_time = temp_file.stat().st_mtime
                    if file_time < cutoff_time:
                        if self.cleanup_file(temp_file):
                            results['success'] += 1
                        else:
                            results['failed'] += 1
                    else:
                        results['skipped'] += 1
                except OSError:
                    results['failed'] += 1
        
        return results
    
    def get_operation_summary(self):
        """Get summary of all operations"""
        total = len(self.operations_log)
        successful = sum(1 for op in self.operations_log if op['success'])
        failed = total - successful
        
        methods = {}
        for op in self.operations_log:
            method = op.get('method', 'unknown')
            methods[method] = methods.get(method, 0) + 1
        
        return {
            'total_operations': total,
            'successful': successful,
            'failed': failed,
            'success_rate': (successful / total * 100) if total > 0 else 0,
            'methods_used': methods
        }

# Demonstrate production file cleanup
print("=== Production File Cleanup Demo ===")

# Initialize cleanup manager
cleanup_manager = FileCleanupManager(use_trash=True, log_operations=True)

# Create test files and directories
test_dir = Path(tempfile.mkdtemp(prefix='cleanup_demo_'))
print(f"Created test directory: {test_dir.name}")

# Create various test files
test_files = [
    test_dir / 'document.txt',
    test_dir / 'image.jpg',
    test_dir / 'data.csv',
    test_dir / 'config.json'
]

for file_path in test_files:
    file_path.write_text(f"Test content for {file_path.name}")

# Create subdirectory with files
subdir = test_dir / 'subdirectory'
subdir.mkdir()
(subdir / 'nested_file.txt').write_text("Nested file content")

print(f"Created {len(test_files)} test files and 1 subdirectory")

# Cleanup operations
print(f"\n=== Cleanup Operations ===")

# Cleanup individual files
for file_path in test_files[:2]:
    success = cleanup_manager.cleanup_file(file_path)
    print(f"Cleanup {file_path.name}: {'✓' if success else '✗'}")
    assert success, f"cleanup of {file_path.name} did not succeed: {cleanup_manager.operations_log[-1]}"

# Cleanup directory with pattern
results = cleanup_manager.cleanup_directory(test_dir, pattern='*.txt')
print(f"Directory cleanup (*.txt): {results['success']} success, {results['failed']} failed")
# document.txt was trashed above, so only subdirectory/nested_file.txt still matches
assert results == {'success': 1, 'failed': 0}, results

# Cleanup remaining files
remaining_results = cleanup_manager.cleanup_directory(test_dir, recursive=True)
print(f"Remaining cleanup: {remaining_results['success']} success, {remaining_results['failed']} failed")
# data.csv and config.json
assert remaining_results == {'success': 2, 'failed': 0}, remaining_results

# Clean up the test directory itself
if test_dir.exists():
    cleanup_manager.cleanup_file(test_dir)

# Show operation summary
print(f"\n=== Operation Summary ===")
summary = cleanup_manager.get_operation_summary()
print(f"Total operations: {summary['total_operations']}")
print(f"Success rate: {summary['success_rate']:.1f}%")
print(f"Methods used: {summary['methods_used']}")
assert summary['failed'] == 0 and not test_dir.exists(), cleanup_manager.operations_log

# Best practices for production
print(f"\n=== Production Best Practices ===")

production_practices = [
    {
        'category': 'Error Handling',
        'practices': [
            'Always wrap send2trash calls in try-except blocks',
            'Provide fallback cleanup methods for critical operations',
            'Log all cleanup operations for audit trails'
        ]
    },
    {
        'category': 'User Experience',
        'practices': [
            'Provide user preference for trash vs permanent deletion',
            'Show confirmation dialogs for bulk operations',
            'Allow users to recover files from trash when possible'
        ]
    },
    {
        'category': 'Performance',
        'practices': [
            'Batch cleanup operations when possible',
            'Implement async cleanup for non-blocking operations',
            'Monitor and limit cleanup operations in tight loops'
        ]
    },
    {
        'category': 'Security',
        'practices': [
            'Validate file paths before cleanup operations',
            'Implement proper permission checks',
            'Sanitize user-provided paths to prevent directory traversal'
        ]
    }
]

for category in production_practices:
    print(f"{category['category']}:")
    for practice in category['practices']:
        print(f"  ✓ {practice}")
    print()

# Integration patterns
print(f"=== Integration Patterns ===")

integration_examples = {
    'Web Applications': 'Cleanup uploaded files and temporary data',
    'Desktop Applications': 'File management and document cleanup',
    'Data Processing': 'Cleanup intermediate files after processing',
    'Development Tools': 'Cleanup build artifacts and temporary files',
    'System Administration': 'Log rotation and cache cleanup'
}

print("Common integration scenarios:")
for scenario, description in integration_examples.items():
    print(f"  {scenario}: {description}")

# Configuration example
print(f"\n=== Configuration Example ===")

config_example = {
    'cleanup': {
        'use_trash': True,
        'log_operations': True,
        'max_retries': 3,
        'batch_size': 100,
        'temp_file_max_age_hours': 24,
        'auto_cleanup_patterns': ['*.tmp', '*.cache', '*.log.old']
    }
}

print("Sample configuration:")
import json
print(json.dumps(config_example, indent=2))

print("Production integration and best practices completed!")